# U9-02 · Aprendizaje incremental y búsqueda de hiperparámetros

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 9, apartados 9.6 y 9.7*

Ejecuta cada celda con **Mayús + Intro**. Para que funcione en el navegador, la flota simulada tiene **50 máquinas** en lugar de 200 (438 000 lecturas en lugar de 1 752 000): los tamaños y tiempos serán menores que en el libro y los resultados, parecidos pero no idénticos.

In [ ]:
%pip install -q duckdb pyarrow
import duckdb, pyarrow
print("DuckDB", duckdb.__version__, "· PyArrow", pyarrow.__version__)

## 0. Los datos simulados

El generador de la unidad (archivo `lecturas.py` del libro), con 50 máquinas. Cada hora registra temperatura, vibración, consumo y horas desde la revisión; desde octubre, la mitad de la flota mide la temperatura 3 °C de más.

In [ ]:
import numpy as np
import pandas as pd

def generar_lecturas(maquinas=50, horas=24 * 365, semilla=0):
    """Lecturas horarias simuladas de una flota durante un año."""
    rng = np.random.default_rng(semilla)
    ritmo = rng.uniform(0.5, 1.5, maquinas)      # desgaste por hora
    desgaste = rng.uniform(0, 300, maquinas)     # desde la revisión
    filas = []
    for h in range(horas):
        desgaste += ritmo
        temp = 60 + 0.01 * desgaste + rng.normal(0, 2, maquinas)
        if h >= 24 * 273:              # desde octubre, media flota
            temp[: maquinas // 2] += 3  # con sensor descalibrado
        vib = 2 + 0.004 * desgaste + rng.normal(0, 0.5, maquinas)
        kw = 40 + 0.5 * (temp - 60) + rng.normal(0, 3, maquinas)
        z = -9 + 0.006 * desgaste + 0.8 * (vib - 2)
        averia = rng.random(maquinas) < 1 / (1 + np.exp(-z))
        filas.append((np.full(maquinas, h), np.arange(maquinas),
                      temp, vib, kw, desgaste.copy(), averia))
        desgaste[averia] = 0                      # se repara
    h, m, t, v, k, d, a = map(np.concatenate, zip(*filas))
    df = pd.DataFrame({
        "fecha": pd.Timestamp("2025-01-01") + pd.to_timedelta(h, "h"),
        "maquina": m, "temperatura": t, "vibracion": v,
        "consumo_kw": k, "horas_desde_revision": d, "averia": a})
    # objetivo: ¿se averiará la máquina en las próximas 24 horas?
    df = df.sort_values(["maquina", "fecha"], ignore_index=True)
    def en_24h(s):                               # ¿avería en las 24 h
        futuro = s[::-1].rolling(24, 1).max()[::-1]   # siguientes?
        return futuro.shift(-1, fill_value=0)
    proxima = df.groupby("maquina")["averia"].transform(en_24h)
    df["averia_24h"] = proxima.astype(bool)
    return df

In [ ]:
lecturas = generar_lecturas()
lecturas = lecturas.astype({"maquina": "int16", "temperatura": "float32",
    "vibracion": "float32", "consumo_kw": "float32",
    "horas_desde_revision": "float32"})
lecturas.to_parquet("lecturas.parquet", index=False)
print(len(lecturas), "lecturas guardadas en lecturas.parquet")

## 1. Aprender mes a mes con `partial_fit` (ejemplo 3)

In [ ]:
import time
import pandas as pd
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

X_COLS = ["temperatura", "vibracion", "consumo_kw",
          "horas_desde_revision"]
datos = pd.read_parquet("lecturas.parquet")
mes = datos["fecha"].dt.month
test = datos[mes.between(7, 9)]
X_test, y_test = test[X_COLS], test["averia_24h"]

# 1) aprendizaje incremental: un mes cada vez
escala = StandardScaler()
for m in range(1, 7):                      # primera pasada: escala
    escala.partial_fit(datos.loc[mes == m, X_COLS])
sgd = SGDClassifier(loss="log_loss", random_state=0)
t0 = time.perf_counter()
for m in range(1, 7):
    trozo = datos[mes == m]
    sgd.partial_fit(escala.transform(trozo[X_COLS]),
                    trozo["averia_24h"], classes=[False, True])
    p = sgd.predict_proba(escala.transform(X_test))[:, 1]
    print(f"tras el mes {m}: AUC = {roc_auc_score(y_test, p):.4f}")
print(f"SGD: {time.perf_counter() - t0:.2f} s")

# 2) el mismo modelo con todos los datos en memoria
ent = datos[mes <= 6]
t0 = time.perf_counter()
rl = LogisticRegression(max_iter=1000).fit(
    escala.transform(ent[X_COLS]), ent["averia_24h"])
p = rl.predict_proba(escala.transform(X_test))[:, 1]
print(f"Logística completa: AUC = {roc_auc_score(y_test, p):.4f}"
      f" ({time.perf_counter() - t0:.2f} s)")

**Prueba.** Cambia el orden de los meses (por ejemplo, de junio a enero). ¿Cambia el resultado final?

## 2. Curva de aprendizaje (ejemplo 4)

In [ ]:
import time
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

X_COLS = ["temperatura", "vibracion", "consumo_kw",
          "horas_desde_revision"]
datos = pd.read_parquet("lecturas.parquet")
mes = datos["fecha"].dt.month
ent, test = datos[mes <= 6], datos[mes.between(7, 9)]
modelos = {
    "logística": make_pipeline(StandardScaler(),
                               LogisticRegression(max_iter=1000)),
    "boosting": HistGradientBoostingClassifier(random_state=0)}
for n in [1_000, 10_000, 50_000, len(ent)]:
    muestra = ent.sample(n, random_state=0)
    for nombre, modelo in modelos.items():
        t0 = time.perf_counter()
        modelo.fit(muestra[X_COLS], muestra["averia_24h"])
        seg = time.perf_counter() - t0
        p = modelo.predict_proba(test[X_COLS])[:, 1]
        auc = roc_auc_score(test["averia_24h"], p)
        print(f"{n:>7} filas  {nombre:<9} AUC = {auc:.4f}"
              f"  ({seg:.2f} s)")

## 3. Tres búsquedas (ejemplo 5)

Versión reducida para el navegador: 24 combinaciones y 20 000 lecturas. Tarda unos minutos.

In [ ]:
import time
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import (GridSearchCV,
    RandomizedSearchCV, HalvingRandomSearchCV)
from sklearn.metrics import roc_auc_score

X_COLS = ["temperatura", "vibracion", "consumo_kw",
          "horas_desde_revision"]
datos = pd.read_parquet("lecturas.parquet")
mes = datos["fecha"].dt.month
ent = datos[mes <= 6].sample(20_000, random_state=0)
test = datos[mes.between(7, 9)]
rejilla = {"learning_rate": [0.03, 0.1, 0.3],
           "max_leaf_nodes": [7, 31],
           "min_samples_leaf": [20, 200],
           "l2_regularization": [0.0, 1.0]}       # 24 combinaciones
base = HistGradientBoostingClassifier(random_state=0)
busquedas = {
    "rejilla": GridSearchCV(base, rejilla, cv=3,
                            scoring="roc_auc"),
    "aleatoria": RandomizedSearchCV(base, rejilla, n_iter=6,
                    cv=3, scoring="roc_auc", random_state=0),
    "por rondas": HalvingRandomSearchCV(base, rejilla,
                    n_candidates=24, factor=3,
                    min_resources=2000, cv=3,
                    scoring="roc_auc", random_state=0)}
for nombre, b in busquedas.items():
    t0 = time.perf_counter()
    b.fit(ent[X_COLS], ent["averia_24h"])
    seg = time.perf_counter() - t0
    p = b.predict_proba(test[X_COLS])[:, 1]
    auc = roc_auc_score(test["averia_24h"], p)
    print(f"{nombre:<10} {seg:6.1f} s  AUC val = {b.best_score_:.4f}"
          f"  AUC test = {auc:.4f}")
    print("           ", b.best_params_)

**Ejercicio.** Añade `n_jobs=-1` a las búsquedas y compara los tiempos (en el navegador no cambiará: solo hay un núcleo).

> Bottou, L. (2010). Large-scale machine learning with stochastic gradient descent. En *Proceedings of COMPSTAT'2010* (pp. 177–186). Physica-Verlag. https://doi.org/10.1007/978-3-7908-2604-3_16